# Academic Performance Prediction Using Supervised Learning
## Notebook 4 of 4 — End-to-End Demo & PDF Report

Runs the two saved models on real students and produces the PDF prediction report that the Streamlit app issues after every prediction (reviewer feedback from Final Review 1).

**Pipeline:** student features → dropout risk model → risk level → exam score model (uses the predicted risk) → PDF report.

> Run notebooks 2 and 3 first so `models/*.pkl` exist.

In [1]:
import numpy as np
import pandas as pd

from src.config import EXAM_MODEL_PATH, DROPOUT_MODEL_PATH, RISK_BANDS, REPORTS_DIR
from src.data import load_raw, risk_label
from src.inference import load_bundle, predict_dropout, predict_exam_score
from src.report import build_report, score_band

dropout_bundle = load_bundle(DROPOUT_MODEL_PATH)
exam_bundle = load_bundle(EXAM_MODEL_PATH)

print("dropout model :", dropout_bundle["model_name"])
print("exam model    :", exam_bundle["model_name"])
print("risk bands    :", RISK_BANDS)

dropout model : Hybrid RF-XGBoost (Soft Voting)
exam model    : Hybrid RF-XGBoost (Stacking)
risk bands    : {'High': 0.66, 'Medium': 0.33}


### 1. Predict for five sample students

In [2]:
students = load_raw().head(5)

rows = []
for _, row in students.iterrows():
    record = row.to_dict()

    dropout = predict_dropout(dropout_bundle, record)
    level = risk_label(dropout["probability"], RISK_BANDS)

    exam_record = dict(record, dropout_risk=dropout["label"])
    score = predict_exam_score(exam_bundle, exam_record)

    rows.append({
        "student_id": int(record["student_id"]),
        "actual_risk": record["dropout_risk"],
        "pred_dropout": dropout["label"],
        "P(dropout)": round(dropout["probability"], 3),
        "risk_level": level,
        "actual_score": record["exam_score"],
        "pred_score": round(score, 1),
        "band": score_band(score),
    })

pd.DataFrame(rows)

,student_id,actual_risk,pred_dropout,P(dropout),risk_level,actual_score,pred_score,band
0,100000,No,No,0.0,Low,100,97.7,Excellent
1,100001,No,No,0.0,Low,99,97.8,Excellent
2,100002,No,No,0.0,Low,98,93.8,Excellent
3,100003,No,No,0.0,Low,100,97.7,Excellent
4,100004,No,No,0.0,Low,98,98.0,Excellent


### 2. Generate the PDF report for one student

In [3]:
record = students.iloc[0].to_dict()

dropout = predict_dropout(dropout_bundle, record)
level = risk_label(dropout["probability"], RISK_BANDS)
score = predict_exam_score(exam_bundle, dict(record, dropout_risk=dropout["label"]))

pdf_bytes = build_report(record, dropout, level, score)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
pdf_path = REPORTS_DIR / f"sample_report_{int(record['student_id'])}.pdf"
pdf_path.write_bytes(pdf_bytes)

print(f"dropout risk : {dropout['label']}  (P = {dropout['probability']:.1%}, {level})")
print(f"exam score    : {score:.1f} / 100  ({score_band(score)})")
print(f"report saved : {pdf_path}  ({len(pdf_bytes):,} bytes)")
print(f"header       : {pdf_bytes[:5]}")

dropout risk : No  (P = 0.0%, Low)
exam score    : 97.7 / 100  (Excellent)
report saved : D:\Downloads\freebuff\reports\sample_report_100000.pdf  (4,551 bytes)
header       : b'%PDF-'


### 3. Run the web application

The same models power the Streamlit app with the two roles required by the reviewers:

```bash
streamlit run app.py
```

| Role | Can do |
|---|---|
| **Teacher** | enter student details → dropout risk meter + predicted score → download the PDF report |
| **Student** | view an existing student's report only (no prediction form) |

Open it from VS Code's terminal (**Terminal → New Terminal**, or `Ctrl+`` `) and run the command above.